# 02. Baseline modeliai (UCI HAR)

Šiame etape **nekuriame** SVM, MLP, SOM ir nedarome abliacijos ar triukšmo bandymų. Tikslas — turėti paprastus palyginimo taškus prieš sudėtingesnius metodus.

Naudojamas **tas pats** 1 etape išsaugotas subject-disjoint TRAIN/TEST skaidymas (`results/subject_disjoint_split.npz`). Naujo skaidymo nekuriame.

## Kas yra baseline ir kam jis reikalingas

**Baseline** — paprastas, lengvai paaiškinamas modelis. Jis nėra galutinis sprendimas, o atskaitos taškas: jei vėlesnis metodas jo nepralenktų, nauda būtų abejotina.

**Majority Class baseline** visada spėja dažniausią TRAIN klasę. Jis parodo, kokį Macro-F1 ir Balanced Accuracy gautume *nieko neišmokę* iš požymių. Jei k-NN (ar vėliau SVM/MLP) būtų tik nedaug geresnis už šį dummy, modelis beveik nepadėtų.

k-NN yra antrasis baseline: jis jau naudoja 561 požymį, bet vis dar yra paprastas atstumų klasifikatorius be sudėtingo mokymo.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"RANDOM_STATE = {RANDOM_STATE}")

RANDOM_STATE = 42


## TRAIN/TEST užkrovimas

Užkrauname tik 1 etape išsaugotą failą. Hiperparametrų paieškai naudosime **tik TRAIN**. Galutinis TEST paliekamas vienam įvertinimui po to, kai parametrai jau parinkti.

In [2]:
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
project_root = None
for cand in candidates:
    if (cand / "results").exists() and (cand / "notebooks").exists():
        project_root = cand
        break
if project_root is None:
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

split_path = project_root / "results" / "subject_disjoint_split.npz"
if not split_path.exists():
    raise FileNotFoundError(
        f"Nerastas {split_path}. Pirmiausia paleiskite notebooks/01_data_preparation.ipynb."
    )

split = np.load(split_path, allow_pickle=True)

X_train = split["X_train_final"]
X_test = split["X_test_final"]
y_train = split["y_train_final"]
y_test = split["y_test_final"]
subjects_train = split["subjects_train_final"]
subjects_test = split["subjects_test_final"]
activity_ids = split["activity_ids"]
activity_names = split["activity_names"]
saved_random_state = int(split["random_state"][0])

id_to_name = {
    int(i): str(n) for i, n in zip(activity_ids, activity_names)
}

print(f"Projekto šaknis: {project_root}")
print(f"Skaidymas: {split_path}")
print(f"Išsaugotas RANDOM_STATE: {saved_random_state}")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)
print("TRAIN subject skaičius:", len(np.unique(subjects_train)))
print("TEST subject skaičius:", len(np.unique(subjects_test)))
print(
    "TRAIN ∩ TEST subject:",
    sorted(set(subjects_train.tolist()) & set(subjects_test.tolist())),
)

Projekto šaknis: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26
Skaidymas: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\subject_disjoint_split.npz
Išsaugotas RANDOM_STATE: 42
X_train: (7144, 561) y_train: (7144,)
X_test: (3155, 561) y_test: (3155,)
TRAIN subject skaičius: 21
TEST subject skaičius: 9
TRAIN ∩ TEST subject: []


## Majority Class baseline

Dažniausią klasę skaičiuojame **tik iš TRAIN**. Tada visiems TEST įrašams priskiriame tą pačią klasę. Taip TEST informacija nepatenka į baseline taisyklę.

In [3]:
values, counts = np.unique(y_train, return_counts=True)
majority_class = int(values[np.argmax(counts)])
majority_name = id_to_name[majority_class]

y_pred_majority = np.full(shape=y_test.shape, fill_value=majority_class, dtype=y_test.dtype)

majority_macro_f1 = f1_score(y_test, y_pred_majority, average="macro")
majority_bal_acc = balanced_accuracy_score(y_test, y_pred_majority)

print("TRAIN klasių dažniai:")
for cls, cnt in zip(values, counts):
    print(f"  {int(cls)} ({id_to_name[int(cls)]}): {int(cnt)}")
print()
print(f"Majority klasė (iš TRAIN): {majority_class} ({majority_name})")
print(f"Majority TEST Macro-F1: {majority_macro_f1:.6f}")
print(f"Majority TEST Balanced Accuracy: {majority_bal_acc:.6f}")

TRAIN klasių dažniai:
  1 (WALKING): 1214
  2 (WALKING_UPSTAIRS): 1060
  3 (WALKING_DOWNSTAIRS): 970
  4 (SITTING): 1233
  5 (STANDING): 1322
  6 (LAYING): 1345

Majority klasė (iš TRAIN): 6 (LAYING)
Majority TEST Macro-F1: 0.053188
Majority TEST Balanced Accuracy: 0.166667


## k-NN: standartizavimas, Pipeline ir group-aware CV

**Kodėl k-NN reikia požymių standartizavimo.** k-NN sprendžia pagal atstumą požymių erdvėje. Jei vieno požymio skalė yra daug didesnė už kito, tas požymis dominuoja atstumą. `StandardScaler` visus požymius perkelia į panašų mastelį (vidurkis 0, dispersija 1), todėl atstumas tampa prasmingesnis.

**Kodėl `StandardScaler` yra Pipeline viduje.** Scalerio vidurkiai ir dispersijos turi būti skaičiuojami tik iš tos dalies, kuria modelis tuo metu mokomas. Jei scalerį pritaikytume visam TRAIN prieš cross-validation, validavimo fold'o statistika nutekėtų į mokymą (**data leakage**). Pipeline kiekviename fold'e scalerį moko iš naujo tik iš CV mokymo dalies.

**Kodėl galutinis TEST negali būti naudojamas hiperparametrų paieškai.** Jei `n_neighbors` ar `weights` rinktumėmės žiūrėdami į TEST rezultatus, TEST taptų slaptu mokymu. Tuomet TEST metrika būtų pernelyg optimistiška ir nebeatspindėtų nematyto žmogaus atpažinimo.

**Kodėl CV turi būti group-aware pagal subject ID.** Vieno žmogaus įrašai yra panašūs. Jei to paties subject įrašai patektų ir į CV mokymą, ir į validavimą, modelis „pamatyti“ tą žmogų ir CV balas būtų per geras. `GroupKFold` visus vieno subject įrašus laiko vienoje fold pusėje.

In [4]:
knn_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier()),
    ]
)

param_grid = {
    "knn__n_neighbors": [3, 5, 7, 11],
    "knn__weights": ["uniform", "distance"],
}

cv = GroupKFold(n_splits=5)

grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

grid_search.fit(X_train, y_train, groups=subjects_train)

cv_table = pd.DataFrame(grid_search.cv_results_)[
    [
        "param_knn__n_neighbors",
        "param_knn__weights",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].sort_values("rank_test_score")
cv_table = cv_table.rename(
    columns={
        "param_knn__n_neighbors": "n_neighbors",
        "param_knn__weights": "weights",
        "mean_test_score": "CV_MacroF1_mean",
        "std_test_score": "CV_MacroF1_std",
        "rank_test_score": "rank",
    }
)

best_n_neighbors = int(grid_search.best_params_["knn__n_neighbors"])
best_weights = str(grid_search.best_params_["knn__weights"])
best_cv_macro_f1 = float(grid_search.best_score_)

print("GroupKFold n_splits = 5, scoring = Macro-F1")
print("Parametrų tinklelis tik TRAIN, groups = subjects_train_final")
print()
print(cv_table.to_string(index=False))
print()
print(f"Geriausi parametrai: n_neighbors={best_n_neighbors}, weights={best_weights}")
print(f"TRAIN CV Macro-F1 (vidurkis): {best_cv_macro_f1:.6f}")

GroupKFold n_splits = 5, scoring = Macro-F1
Parametrų tinklelis tik TRAIN, groups = subjects_train_final

 n_neighbors  weights  CV_MacroF1_mean  CV_MacroF1_std  rank
          11 distance         0.889029        0.013165     1
          11  uniform         0.887956        0.013035     2
           7 distance         0.882454        0.011838     3
           7  uniform         0.879593        0.013053     4
           5 distance         0.877101        0.009919     5
           5  uniform         0.876079        0.011535     6
           3 distance         0.866235        0.011987     7
           3  uniform         0.865874        0.010873     8

Geriausi parametrai: n_neighbors=11, weights=distance
TRAIN CV Macro-F1 (vidurkis): 0.889029


## Galutinis k-NN: mokymas visu TRAIN ir įvertinimas TEST

`GridSearchCV(refit=True)` po paieškos dar kartą apmoko geriausią Pipeline visu TRAIN. Žemiau tą patį darome aiškiai: naujas Pipeline su rastais parametrais, `fit` tik su TRAIN, tada prognozė tik TEST.

In [5]:
final_knn = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "knn",
            KNeighborsClassifier(
                n_neighbors=best_n_neighbors,
                weights=best_weights,
            ),
        ),
    ]
)
final_knn.fit(X_train, y_train)
y_pred_knn = final_knn.predict(X_test)

knn_macro_f1 = f1_score(y_test, y_pred_knn, average="macro")
knn_bal_acc = balanced_accuracy_score(y_test, y_pred_knn)

labels = [int(i) for i in activity_ids]
cm = confusion_matrix(y_test, y_pred_knn, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"tikra_{id_to_name[i]}" for i in labels],
    columns=[f"pred_{id_to_name[i]}" for i in labels],
)

print(f"Galutinis k-NN: n_neighbors={best_n_neighbors}, weights={best_weights}")
print(f"k-NN TEST Macro-F1: {knn_macro_f1:.6f}")
print(f"k-NN TEST Balanced Accuracy: {knn_bal_acc:.6f}")
print()
print("Confusion matrix (eilutės = tikros klasės, stulpeliai = prognozės):")
print(cm_df.to_string())

Galutinis k-NN: n_neighbors=11, weights=distance
k-NN TEST Macro-F1: 0.844324
k-NN TEST Balanced Accuracy: 0.842074

Confusion matrix (eilutės = tikros klasės, stulpeliai = prognozės):
                          pred_WALKING  pred_WALKING_UPSTAIRS  pred_WALKING_DOWNSTAIRS  pred_SITTING  pred_STANDING  pred_LAYING
tikra_WALKING                      483                     23                        2             0              0            0
tikra_WALKING_UPSTAIRS              33                    450                        1             0              0            0
tikra_WALKING_DOWNSTAIRS            48                     86                      302             0              0            0
tikra_SITTING                        0                      1                        0           425            118            0
tikra_STANDING                       0                      0                        0            74            510            0
tikra_LAYING                         0   

## Rezultatų lentelė

Abi eilutės — **TEST** metrikos. Majority Class neturi hiperparametrų. k-NN parametrai parinkti tik TRAIN GroupKFold pagal Macro-F1.

In [6]:
results_table = pd.DataFrame(
    [
        {
            "Model": "Majority Class",
            "Macro-F1": majority_macro_f1,
            "Balanced Accuracy": majority_bal_acc,
        },
        {
            "Model": "k-NN",
            "Macro-F1": knn_macro_f1,
            "Balanced Accuracy": knn_bal_acc,
        },
    ]
)

print(results_table.to_string(index=False))
print()
print(
    "Papildoma k-NN informacija (ne TEST paieška): "
    f"n_neighbors={best_n_neighbors}, weights={best_weights}, "
    f"TRAIN CV Macro-F1={best_cv_macro_f1:.6f}"
)

         Model  Macro-F1  Balanced Accuracy
Majority Class  0.053188           0.166667
          k-NN  0.844324           0.842074

Papildoma k-NN informacija (ne TEST paieška): n_neighbors=11, weights=distance, TRAIN CV Macro-F1=0.889029


## Rezultatų išsaugojimas

Lentelė rašoma į `results/baseline_results.csv`. Papildomai išsaugomi k-NN CV rezultatai ir confusion matrix. Failas `subject_disjoint_split.npz` **nekeičiamas**.

In [7]:
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

table_path = results_dir / "baseline_results.csv"
cv_path = results_dir / "knn_cv_results.csv"
cm_path = results_dir / "knn_confusion_matrix.csv"

results_table.to_csv(table_path, index=False)
cv_table.to_csv(cv_path, index=False)
cm_df.to_csv(cm_path)

print(f"Išsaugota: {table_path}")
print(f"Išsaugota: {cv_path}")
print(f"Išsaugota: {cm_path}")
print("subject_disjoint_split.npz nekeistas.")
print("2 etapas baigtas: Majority Class ir k-NN baseline įvertinti.")

Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\baseline_results.csv
Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\knn_cv_results.csv
Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\knn_confusion_matrix.csv
subject_disjoint_split.npz nekeistas.
2 etapas baigtas: Majority Class ir k-NN baseline įvertinti.
